# Can we predict late delivery?
Target: `late_delivery`. Data: 8,051 consignments dispatched 2024-03 to 2026-02.

Plan: audit the columns for leakage, build features known **at dispatch**, validate with a **time-ordered split** (that is how the model would be used), and compare against naive baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width',140)
DATA='/private/tmp/waldean-agent-baseline/runs-protocol/shipments-claude-A-0/'
df=pd.read_csv(DATA+'data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(pd.read_csv(DATA+'columns.csv').to_string())
print(df.shape, df.isna().sum().sum(),'missing;', df.consignment.duplicated().sum(),'duplicate ids')
df.describe().T

## 1. Leakage check: `claim_paid_eur`
A claim is *settled after* the consignment is delivered, so it is not known at dispatch. Let's see how it relates to the target.

In [ ]:
print(df.groupby(df.claim_paid_eur>0).late_delivery.agg(['mean','count']))
print('AUC of claim_paid_eur alone:', round(roc_auc_score(df.late_delivery, df.claim_paid_eur),3))

Claims are paid on ~90% of late consignments and on **none** of the on-time ones. It is a consequence of lateness, not a predictor, so it is **excluded** from the real model. (It is added once in section 5 only to show how inflated the score would be.)

## 2. Target over time and by carrier

In [ ]:
m=df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.mean()
ax=m.plot(marker='o',figsize=(10,3.5)); ax.set_ylabel('late rate'); ax.set_title('Monthly late-delivery rate'); plt.show()
print(df.groupby('carrier').late_delivery.agg(['mean','count']))

The late rate is **not stable**: it swings seasonally (peak around July 2025) and then collapses to ~1-2% from Oct 2025 onward. Something changed in the process (or in how lateness is recorded) that the columns don't explain. This matters for validation: a random split would mix periods and look better than reality.

## 3. Features available at dispatch

In [ ]:
def feats(d):
    X=pd.DataFrame({'distance_km':d.distance_km,'weight_kg':d.weight_kg,'parcels':d.parcels,
        'kg_per_parcel':d.weight_kg/d.parcels,
        'hour':d.dispatched_on.dt.hour,'dow':d.dispatched_on.dt.dayofweek,
        'weekend':(d.dispatched_on.dt.dayofweek>=5).astype(int),'month':d.dispatched_on.dt.month})
    return pd.concat([X,pd.get_dummies(d.carrier,prefix='c',dtype=int)],axis=1)
X=feats(df); y=df.late_delivery
print(X.assign(y=y).corr()['y'].drop('y').round(3).sort_values())

## 4. Time-ordered evaluation
Train on the first 70% of the timeline, validate on the next 15%, test on the last 15%. Metrics: ROC AUC and average precision (AP; a random model's AP equals the period's late rate).

In [ ]:
n=len(df); i1,i2=int(.7*n),int(.85*n)
sl={'train':slice(0,i1),'val':slice(i1,i2),'test':slice(i2,n)}
for k,s in sl.items(): print(k, df.dispatched_on[s].min().date(),'->',df.dispatched_on[s].max().date(),'late rate',round(y.iloc[s].mean(),3))
def run(cols, name, model):
    model.fit(X.iloc[sl['train']][cols], y.iloc[sl['train']])
    r={'model':name}
    for k in ['val','test']:
        p=model.predict_proba(X.iloc[sl[k]][cols])[:,1]
        r[k+'_auc']=roc_auc_score(y.iloc[sl[k]],p); r[k+'_ap']=average_precision_score(y.iloc[sl[k]],p)
    return r
base=[c for c in X.columns if c!='month']   # month cannot extrapolate across years
hgb=lambda:HistGradientBoostingClassifier(max_depth=3,learning_rate=.05,max_iter=200,random_state=0)
res=[run(['c_coastal','c_express','c_north'],'carrier only',LogisticRegression()),
     run(base,'logreg',make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))),
     run(base,'grad. boosting',hgb())]
pd.DataFrame(res).set_index('model').round(3)

## 5. Contrast: random split, and the leaky variant

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
cv=StratifiedKFold(5,shuffle=True,random_state=0)
p=cross_val_predict(hgb(),X[base],y,cv=cv,method='predict_proba')[:,1]
print('random 5-fold, legit features    AUC',round(roc_auc_score(y,p),3))
p=cross_val_predict(hgb(),X[base].assign(claim=df.claim_paid_eur),y,cv=cv,method='predict_proba')[:,1]
print('random 5-fold, WITH claim (leak) AUC',round(roc_auc_score(y,p),3))

## 6. What drives the predictions? (permutation importance, validation period)

In [ ]:
from sklearn.inspection import permutation_importance
m=hgb().fit(X.iloc[sl['train']][base],y.iloc[sl['train']])
pi=permutation_importance(m,X.iloc[sl['val']][base],y.iloc[sl['val']],scoring='roc_auc',n_repeats=10,random_state=0)
pd.Series(pi.importances_mean,base).sort_values().plot.barh(title='Permutation importance (val AUC drop)'); plt.show()

## 7. Conclusions
(Numbers below are from my run; re-run the notebook to reproduce.)

* **`claim_paid_eur` must not be used.** Alone it gives AUC 0.99 and lifts any model to 0.99, but it is only known after the delivery is late. Using it would look like a triumph and be useless in practice.
* **Legitimate, at-dispatch features carry moderate signal.** ROC AUC about 0.70-0.72 (random 5-fold 0.71; time-ordered val 0.72, test 0.68-0.70), against 0.5 for chance. Longer distance and heavier consignments are later; `coastal` is the worst carrier. Gradient boosting is no better than logistic regression, so the relationship is simple.
* **Good for ranking risk, not for confident individual calls.** AP in the val period is about 0.35 against a 0.16 base rate (roughly 2x lift).
* **Biggest caveat: the late rate is non-stationary.** It falls from ~26% (train) to ~16% (val) to ~1.7% (last 4 months). Test has only ~20 late consignments, so test metrics are very noisy and AP there (0.05 vs 0.017 base) mostly reflects the base rate. Before deploying, find out what changed around Oct 2025 (process change? change in how lateness is recorded?). A model trained on the old regime will over-predict lateness even if its ranking holds.
* **Missing drivers.** Weather, depot load, promised lead time and destination/route are not in the data; they likely limit the score.

Suggested next steps: explain the Oct 2025 shift, add promised-date and route/load features, recalibrate on recent data only, and track precision at a fixed alert volume.
